# Question 1: Speed of Light (Michelson's 1879 Experiment)

In [ ]:
# (a) Load Michelson's 1879 dataset and state the hypothesis testing framework
import json
import numpy as np
import pandas as pd
from scipy import stats

# Load experiment determinations from the JSON file
with open('michelson_speed_of_light.json', 'r') as json_file:
    michelson_payload = json.load(json_file)

# Extract individual speed measurements (in km/s)
michelson_velocities = pd.Series(
    [entry['measured_speed_kmpersec'] for entry in michelson_payload['observations']],
    name='speed_km_s'
)

# Reference benchmark: true speed of light in air (km/s)
accepted_c_air = 299734.5
significance_alpha = 0.05

print(f"Sample size (n): {len(michelson_velocities)}")
print(f"Accepted benchmark (speed of light in air): {accepted_c_air:,.1f} km/s")
print(f"Significance threshold (alpha): {significance_alpha}")

In [ ]:
# (b) Assess normality of the sample using the Shapiro-Wilk test

# Perform the Shapiro-Wilk test on the 20 speed determinations
shapiro_w, shapiro_p = stats.shapiro(michelson_velocities)

print(f"Sample size (n): {len(michelson_velocities)}")
print(f"Shapiro-Wilk statistic (W): {shapiro_w:.4f}")
print(f"Shapiro-Wilk p-value: {shapiro_p:.4f}")

In [ ]:
# (c) Conduct one-sample t-test against accepted speed of light in air

# Compute sample metrics
sample_count = len(michelson_velocities)
observed_mean = michelson_velocities.mean()
benchmark_mean = accepted_c_air
observed_sd = michelson_velocities.std(ddof=1)
standard_error = observed_sd / np.sqrt(sample_count)
degrees_of_freedom = sample_count - 1

# Execute one-sample t-test
t_statistic, two_tailed_p = stats.ttest_1samp(michelson_velocities, benchmark_mean)

# Display required statistical metrics
print(f"Sample size (n): {sample_count}")
print(f"Sample mean: {observed_mean:,.2f} km/s")
print(f"Population mean (accepted value): {benchmark_mean:,.2f} km/s")
print(f"Sample standard deviation: {observed_sd:,.2f} km/s")
print(f"Standard error of the mean (SEM): {standard_error:,.2f} km/s")
print(f"Degrees of freedom (df): {degrees_of_freedom}")
print(f"t-statistic: {t_statistic:.4f}")
print(f"p-value: {two_tailed_p:.4e}")
print(f"Reject H0 at alpha = {significance_alpha}: {two_tailed_p < significance_alpha}")

In [ ]:
# (d) Calculate bias and evaluate counterfactual scenario with n = 200

# 1. Bias calculations
measured_bias = observed_mean - benchmark_mean
percentage_bias = (measured_bias / benchmark_mean) * 100

print(f"Absolute bias: {measured_bias:+,.2f} km/s")
print(f"Percentage bias: {percentage_bias:+.4f}%")

# 2. Counterfactual recomputation with n = 200 (same mean and SD)
hypothetical_size = 200
scaled_sem = observed_sd / np.sqrt(hypothetical_size)
scaled_df = hypothetical_size - 1
scaled_t = measured_bias / scaled_sem
scaled_p = 2 * (1 - stats.t.cdf(abs(scaled_t), df=scaled_df))

print(f"Counterfactual SEM (n = {hypothetical_size}): {scaled_sem:,.2f} km/s")
print(f"Counterfactual df: {scaled_df}")
print(f"Counterfactual t-statistic: {scaled_t:.4f}")
print(f"Counterfactual p-value: {scaled_p:.4e}")

# Question 2: Chick Weights by Feed Type (Snedecor, 1948)

In [ ]:
# (a) State hypothesis testing framework and justify test selection
feed_alpha = 0.05
print(f"Significance threshold (alpha): {feed_alpha}")

In [ ]:
# (b) Load chick weight dataset and compute descriptive statistics per feed group

# Load data from CSV
poultry_df = pd.read_csv('chickwts_casein_soybean.csv')

# Separate into series for each diet
casein_group = poultry_df[poultry_df['feed'] == 'casein']['chick_weight_g']
soybean_group = poultry_df[poultry_df['feed'] == 'soybean']['chick_weight_g']

# Compute sample statistics for casein
n_casein = len(casein_group)
mean_casein = casein_group.mean()
sd_casein = casein_group.std(ddof=1)
var_casein = casein_group.var(ddof=1)

# Compute sample statistics for soybean
n_soybean = len(soybean_group)
mean_soybean = soybean_group.mean()
sd_soybean = soybean_group.std(ddof=1)
var_soybean = soybean_group.var(ddof=1)

# Display summary table
summary_stats = pd.DataFrame({
    'Sample Size (n)': [n_casein, n_soybean],
    'Mean (g)': [round(mean_casein, 2), round(mean_soybean, 2)],
    'Std Deviation (g)': [round(sd_casein, 2), round(sd_soybean, 2)],
    'Variance (g^2)': [round(var_casein, 2), round(var_soybean, 2)]
}, index=['Casein', 'Soybean'])

print(summary_stats.to_string())

In [ ]:
# (c) Carry out Welch's t-test

# Standard error of the difference in means
se_diff = np.sqrt((var_casein / n_casein) + (var_soybean / n_soybean))

# Welch-Satterthwaite degrees of freedom
welch_df_numerator = ((var_casein / n_casein) + (var_soybean / n_soybean)) ** 2
welch_df_denominator = (((var_casein / n_casein) ** 2) / (n_casein - 1)) + (((var_soybean / n_soybean) ** 2) / (n_soybean - 1))
welch_degrees_freedom = welch_df_numerator / welch_df_denominator

# Welch t-test via scipy (equal_var=False specifies Welch's correction)
welch_t_stat, welch_p_val = stats.ttest_ind(casein_group, soybean_group, equal_var=False)

print(f"Difference in means (Casein - Soybean): {mean_casein - mean_soybean:,.2f} g")
print(f"Standard error of difference: {se_diff:,.2f} g")
print(f"Welch degrees of freedom: {welch_degrees_freedom:.2f}")
print(f"Welch t-statistic: {welch_t_stat:.4f}")
print(f"p-value: {welch_p_val:.4f}")
print(f"Reject H0 at alpha = {feed_alpha}: {welch_p_val < feed_alpha}")

In [ ]:
# (d) Evaluate effect size and practical/agricultural importance

weight_diff_g = mean_casein - mean_soybean
pct_gain = (weight_diff_g / mean_soybean) * 100

# Pooled standard deviation for Cohen's d effect size
pooled_sd = np.sqrt(((n_casein - 1) * var_casein + (n_soybean - 1) * var_soybean) / (n_casein + n_soybean - 2))
cohens_d = weight_diff_g / pooled_sd

print(f"Mean weight difference: {weight_diff_g:,.2f} g")
print(f"Percentage difference: {pct_gain:.1f}%")
print(f"Cohen's d: {cohens_d:.2f}")

In [ ]:
# (e) Generate side-by-side boxplot comparing chick weights by feed type
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 5))

# Plot boxplot without version-dependent keyword arguments
plt.boxplot(
    [casein_group, soybean_group],
    patch_artist=True,
    widths=0.45,
    boxprops=dict(facecolor='#d9eaf7', color='#1b4965', linewidth=1.5),
    medianprops=dict(color='#c1121f', linewidth=2.0),
    whiskerprops=dict(color='#1b4965', linewidth=1.5),
    capprops=dict(color='#1b4965', linewidth=1.5),
    flierprops=dict(marker='o', markerfacecolor='#e63946', markersize=6, linestyle='none')
)

# Set category tick labels directly (universal across all Matplotlib versions)
plt.xticks([1, 2], ['Casein', 'Soybean'], fontsize=11)
plt.ylabel('Chick Weight after 6 Weeks (grams)', fontsize=11)
plt.title('Comparison of 6-Week Chick Weights by Feed Type', fontsize=12, fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Question 3: Fertility Rate vs. GDP per Capita (World Bank 2024)

In [ ]:
# (a) Load 2024 World Bank data via API for UN Member States and plot Fertility Rate vs. GDP per Capita (PPP)
import json
import urllib.request
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# 1. Official 193 United Nations Member States (ISO-3 codes)
# Source: United Nations Member States (https://www.un.org/en/about-us/member-states)
un_member_codes = {
    'AFG', 'ALB', 'DZA', 'AND', 'AGO', 'ATG', 'ARG', 'ARM', 'AUS', 'AUT', 'AZE',
    'BHS', 'BHR', 'BGD', 'BRB', 'BLR', 'BEL', 'BLZ', 'BEN', 'BTN', 'BOL', 'BIH',
    'BWA', 'BRA', 'BRN', 'BGR', 'BFA', 'BDI', 'CPV', 'KHM', 'CMR', 'CAN', 'CAF',
    'TCD', 'CHL', 'CHN', 'COL', 'COM', 'COG', 'COD', 'CRI', 'CIV', 'HRV', 'CUB',
    'CYP', 'CZE', 'PRK', 'DNK', 'DJI', 'DMA', 'DOM', 'ECU', 'EGY', 'SLV', 'GNQ',
    'ERI', 'EST', 'SWZ', 'ETH', 'FJI', 'FIN', 'FRA', 'GAB', 'GMB', 'GEO', 'DEU',
    'GHA', 'GRC', 'GRD', 'GTM', 'GIN', 'GNB', 'GUY', 'HTI', 'HND', 'HUN', 'ISL',
    'IND', 'IDN', 'IRN', 'IRQ', 'IRL', 'ISR', 'ITA', 'JAM', 'JPN', 'JOR', 'KAZ',
    'KEN', 'KIR', 'KWT', 'KGZ', 'LAO', 'LVA', 'LBN', 'LSO', 'LBR', 'LBY', 'LIE',
    'LTU', 'LUX', 'MDG', 'MWI', 'MYS', 'MDV', 'MLI', 'MLT', 'MHL', 'MRT', 'MUS',
    'MEX', 'FSM', 'MDA', 'MCO', 'MNG', 'MNE', 'MAR', 'MOZ', 'MMR', 'NAM', 'NRU',
    'NPL', 'NLD', 'NZL', 'NIC', 'NER', 'NGA', 'MKD', 'NOR', 'OMN', 'PAK', 'PLW',
    'PAN', 'PNG', 'PRY', 'PER', 'PHL', 'POL', 'PRT', 'QAT', 'KOR', 'ROU', 'RUS',
    'RWA', 'KNA', 'LCA', 'VCT', 'WSM', 'SMR', 'STP', 'SAU', 'SEN', 'SRB', 'SYC',
    'SLE', 'SGP', 'SVK', 'SVN', 'SLB', 'SOM', 'ZAF', 'SSD', 'ESP', 'LKA', 'SDN',
    'SUR', 'SWE', 'CHE', 'SYR', 'TJK', 'THA', 'TLS', 'TGO', 'TON', 'TTO', 'TUN',
    'TUR', 'TKM', 'TUV', 'UGA', 'UKR', 'ARE', 'GBR', 'TZA', 'USA', 'URY', 'UZB',
    'VUT', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE'
}

assert len(un_member_codes) == 193, 'The UN Member State set must contain 193 codes.'

# 2. Fetch indicators from World Bank API for 2024
def fetch_world_bank_data(indicator_id):
    api_url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator_id}?date=2024&format=json&per_page=300"
    request = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(request) as response:
        content = json.loads(response.read().decode('utf-8'))[1]
    return {
        item['countryiso3code']: (item['country']['value'], item['value'])
        for item in content if item.get('countryiso3code')
    }

# Fetch Total Fertility Rate (SP.DYN.TFRT.IN) and GDP per Capita PPP (NY.GDP.PCAP.PP.CD)
fertility_lookup = fetch_world_bank_data('SP.DYN.TFRT.IN')
gdp_ppp_lookup = fetch_world_bank_data('NY.GDP.PCAP.PP.CD')

# 3. Build clean DataFrame restricted to UN Member States
country_records = []
for code in sorted(un_member_codes):
    country_name = fertility_lookup.get(code, (code, None))[0]
    fertility_val = fertility_lookup.get(code, (None, None))[1]
    gdp_val = gdp_ppp_lookup.get(code, (None, None))[1]
    country_records.append({
        'country_code': code,
        'country_name': country_name,
        'fertility_rate': fertility_val,
        'gdp_per_capita_ppp': gdp_val
    })

wb_demographics = pd.DataFrame(country_records).dropna(subset=['fertility_rate', 'gdp_per_capita_ppp']).reset_index(drop=True)

# 4. Produce carefully labeled scatter plot with logarithmic x-axis
plt.figure(figsize=(8.5, 5.5))
plt.scatter(
    wb_demographics['gdp_per_capita_ppp'],
    wb_demographics['fertility_rate'],
    color='#2a6f97',
    edgecolor='#012a4a',
    alpha=0.75,
    s=50
)

plt.xscale('log')
plt.xlabel('GDP per Capita, PPP (Current International $, Log Scale)', fontsize=11)
plt.ylabel('Total Fertility Rate (Births per Woman)', fontsize=11)
plt.title('Global Fertility Rate vs. GDP per Capita, PPP (2024) — UN Member States', fontsize=12, fontweight='bold')
plt.grid(True, which='both', linestyle='--', alpha=0.45)
plt.tight_layout()
plt.show()

# Display summary count
print(f"UN Member States with complete 2024 data: {len(wb_demographics)}")

In [ ]:
# (b) Estimate Pearson (raw and log GDP) and Spearman correlations

# 1. Natural log transform of GDP per capita
wb_demographics['log_gdp_per_capita_ppp'] = np.log(wb_demographics['gdp_per_capita_ppp'])

# 2. Pearson correlation on raw GDP
pearson_raw_r, pearson_raw_p = stats.pearsonr(
    wb_demographics['fertility_rate'],
    wb_demographics['gdp_per_capita_ppp']
)

# 3. Pearson correlation on log(GDP)
pearson_log_r, pearson_log_p = stats.pearsonr(
    wb_demographics['fertility_rate'],
    wb_demographics['log_gdp_per_capita_ppp']
)

# 4. Spearman rank correlation on raw GDP
spearman_rho, spearman_p = stats.spearmanr(
    wb_demographics['fertility_rate'],
    wb_demographics['gdp_per_capita_ppp']
)

# Display strictly the computed numerical coefficients
print(f"Pearson (Raw GDP):       r = {pearson_raw_r:.4f}  (p = {pearson_raw_p:.4e})")
print(f"Pearson (log GDP):       r = {pearson_log_r:.4f}  (p = {pearson_log_p:.4e})")
print(f"Spearman Rank (Raw GDP): rho = {spearman_rho:.4f}  (p = {spearman_p:.4e})")

In [ ]:
# (c) Identify extreme observations (|z| > 2.5) and recompute correlations

# 1. Compute z-scores for fertility rate and log GDP per capita
wb_demographics['z_fertility'] = stats.zscore(wb_demographics['fertility_rate'])
wb_demographics['z_log_gdp'] = stats.zscore(wb_demographics['log_gdp_per_capita_ppp'])

# 2. Filter countries where |z| > 2.5 on fertility OR log GDP
outlier_mask = (wb_demographics['z_fertility'].abs() > 2.5) | (wb_demographics['z_log_gdp'].abs() > 2.5)
outlier_countries = wb_demographics[outlier_mask].copy()

print(f"Extreme observations identified (|z| > 2.5): {len(outlier_countries)}")
print(outlier_countries[['country_code', 'country_name', 'fertility_rate', 'gdp_per_capita_ppp', 'z_fertility', 'z_log_gdp']].to_string(index=False))

# 3. Create trimmed dataset excluding extreme observations
trimmed_demographics = wb_demographics[~outlier_mask].copy()

# 4. Recompute correlations on trimmed data
trim_pearson_raw_r, trim_pearson_raw_p = stats.pearsonr(
    trimmed_demographics['fertility_rate'],
    trimmed_demographics['gdp_per_capita_ppp']
)

trim_pearson_log_r, trim_pearson_log_p = stats.pearsonr(
    trimmed_demographics['fertility_rate'],
    trimmed_demographics['log_gdp_per_capita_ppp']
)

trim_spearman_rho, trim_spearman_p = stats.spearmanr(
    trimmed_demographics['fertility_rate'],
    trimmed_demographics['gdp_per_capita_ppp']
)

print(f"\nTrimmed sample size: {len(trimmed_demographics)}")
print(f"Trimmed Pearson (Raw GDP):       r = {trim_pearson_raw_r:.4f}  (p = {trim_pearson_raw_p:.4e})")
print(f"Trimmed Pearson (log GDP):       r = {trim_pearson_log_r:.4f}  (p = {trim_pearson_log_p:.4e})")
print(f"Trimmed Spearman Rank (Raw GDP): rho = {trim_spearman_rho:.4f}  (p = {trim_spearman_p:.4e})")